In [ ]:
# FIGURES 3 & 4: Scenario Comparisons (Flux and TCW Differences)

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter, FuncFormatter
import cartopy.crs as ccrs
import xskillscore as xs
import cmocean
import cmcrameri.cm as cmc

# =============================================================================
# 0. JOURNAL FORMATTING SETTINGS
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12 
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['axes.linewidth'] = 1.0 
plt.rcParams['pdf.fonttype'] = 42 
plt.rcParams['ps.fonttype'] = 42

def format_single_decimal(x, pos):
    return f"{x:.1f}"

formatter_1d = FuncFormatter(format_single_decimal)
sf = ScalarFormatter(useOffset=False)
sf.set_scientific(False)

# =============================================================================
# 1. PATH CONFIGURATION & SCENARIOS
# =============================================================================
NOISE_THRESHOLD = 0.0

# Use relative paths assuming this script is run from the repository root
root_path = "./output/"

# ---> A. GLOBAL BASELINE FILES <---
file_base_bwd = os.path.join(root_path, "UTrack_2024_05_bwd_Global_config_bwdmix24_interp.nc")
file_base_fwd = os.path.join(root_path, "UTrack_2024_05_fwd_Global_config_fwdmix24_interp.nc")

# ---> B. MAIN SCENARIO FILES <---
file_scen_main_bwd = os.path.join(root_path, "UTrack_2024_05_bwd_Global_config_bwdmix1_interp.nc")
file_scen_main_fwd = os.path.join(root_path, "UTrack_2024_05_fwd_Global_config_fwdmix1_interp.nc")

# ---> C. EXTRA SCENARIOS <---
# Uncomment and adjust paths if additional scenarios are needed
EXTRA_SCENARIOS = []
# EXTRA_SCENARIOS = [
#     {
#         'label': 'No mix',
#         'file_bwd': os.path.join(root_path, "UTrack_2024_05_bwd_Global_config_bwdNOmix_interp.nc"),
#         'file_fwd': os.path.join(root_path, "UTrack_2024_05_fwd_Global_config_fwdNOmix_interp.nc"),
#         'color': '#D89E4E',   
#         'linestyle': '-'
#     },
#     {
#         'label': 'No w (1h)',
#         'file_bwd': os.path.join(root_path, "UTrack_2024_05_bwd_Global_config_bwdmix1_noW_interp.nc"),
#         'file_fwd': os.path.join(root_path, "UTrack_2024_05_fwd_Global_config_fwdmix1_noW_interp.nc"),
#         'color': '#B200ED',   
#         'linestyle': '-'
#     }
# ]

# Variables
var_flux_u = 'raw_deposition' 
var_flux_e = 'reference_output_ERA5'
var_tcw_l = 'tot_lagrangian_tcw' 
var_tcw_e = 'tot_era5_tcw'
 
LABEL_SCEN_MAIN = "1h mix"
LABEL_BASE = "Standard (q-only)"
LABEL_ERA5 = "ERA5" 

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================
def compute_grid_areas(lats, lons):
    R = 6371000.0 
    lats_np, lons_np = np.array(lats), np.array(lons)
    lat_bounds = np.zeros(len(lats_np) + 1)
    lat_bounds[1:-1] = 0.5 * (lats_np[:-1] + lats_np[1:])
    lat_bounds[0] = lats_np[0] - 0.5 * (lats_np[1] - lats_np[0])
    lat_bounds[-1] = lats_np[-1] + 0.5 * (lats_np[-1] - lats_np[-2])
    lat_bounds_rad = np.radians(lat_bounds)
    
    lon_bounds = np.zeros(len(lons_np) + 1)
    lon_bounds[1:-1] = 0.5 * (lons_np[:-1] + lons_np[1:])
    lon_bounds[0] = lons_np[0] - 0.5 * (lons_np[1] - lons_np[0])
    lon_bounds[-1] = lons_np[-1] + 0.5 * (lons_np[-1] - lons_np[-2])
    lon_bounds_rad = np.radians(lon_bounds)
    
    d_lon = np.abs(lon_bounds_rad[1:] - lon_bounds_rad[:-1]) 
    sin_lat = np.sin(lat_bounds_rad)
    d_sin_lat = np.abs(sin_lat[1:] - sin_lat[:-1])
    area_grid_np = R**2 * d_sin_lat[:, np.newaxis] * d_lon[np.newaxis, :]
    
    return xr.DataArray(area_grid_np, coords={'lat': lats, 'lon': lons}, dims=['lat', 'lon'])

# FLUX Data Preparation
def prep_data_flux(filepath, v_u, v_e):
    with xr.open_dataset(filepath) as ds:
        if 'latitude' in ds.coords: ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})
        area_m2 = compute_grid_areas(ds.lat, ds.lon)
        area_safe = area_m2.where(area_m2 > 1e-6)
        
        map_mm_u = (ds[v_u] / area_safe).where(lambda x: x > NOISE_THRESHOLD, 0.0).fillna(0)
        vol_u = (map_mm_u / 1000.0) * area_m2
        map_mm_e = (ds[v_e] / area_safe).where(lambda x: x > NOISE_THRESHOLD, 0.0).fillna(0)
        vol_e = (map_mm_e / 1000.0) * area_m2
        return map_mm_u, vol_u, map_mm_e, vol_e, area_m2

# TCW Data Preparation
def prep_data_tcw(filepath, v_l, v_e):
    with xr.open_dataset(filepath) as ds:
        if 'latitude' in ds.coords: ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})
        area_m2 = compute_grid_areas(ds.lat, ds.lon)
        area_safe = area_m2.where(area_m2 > 1e-6)
        
        mm_u = (ds[v_l] / area_safe).fillna(0)
        mm_e = (ds[v_e] / area_safe).fillna(0)
        vol_u = (ds[v_l] / 1000.0).fillna(0)
        vol_e = (ds[v_e] / 1000.0).fillna(0)
        return mm_u, vol_u, mm_e, vol_e, area_m2

def get_1d_diffs_with_pole_filter(vol_u, vol_e):
    lat_u_raw, lat_e_raw = vol_u.sum(dim='lon'), vol_e.sum(dim='lon')
    lat_u = lat_u_raw.where(np.abs(lat_u_raw.lat) <= 89, drop=True)
    lat_e = lat_e_raw.where(np.abs(lat_e_raw.lat) <= 89, drop=True)
    
    diff_lat = lat_u - lat_e
    diff_lon = vol_u.sum(dim='lat') - vol_e.sum(dim='lat')
    return diff_lat, diff_lon

def compute_statistics(sim_xr, obs_xr, area_xr):
    weights = area_xr.fillna(0)
    mask = (obs_xr > 0)
    s_masked = sim_xr.where(mask)
    o_masked = obs_xr.where(mask)
    w_masked = weights.where(mask).fillna(0)
    
    sum_w = w_masked.sum().values.item()
    if sum_w == 0: return {'r': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'KGE': np.nan}
    
    r = xs.pearson_r(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    rmse = xs.rmse(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    mae = xs.mae(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    
    mean_s = s_masked.weighted(w_masked).mean(dim=['lat', 'lon'], skipna=True).values.item()
    mean_o = o_masked.weighted(w_masked).mean(dim=['lat', 'lon'], skipna=True).values.item()
    std_s = s_masked.weighted(w_masked).std(dim=['lat', 'lon'], skipna=True).values.item()
    std_o = o_masked.weighted(w_masked).std(dim=['lat', 'lon'], skipna=True).values.item()
    
    if std_o == 0 or mean_o == 0 or np.isnan(r): kge = np.nan
    else: kge = 1 - np.sqrt((r - 1)**2 + (std_s / std_o - 1)**2 + (mean_s / mean_o - 1)**2)
    return {'r': r, 'RMSE': rmse, 'MAE': mae, 'KGE': kge}

def get_multi_envelope_limits(list_of_data_arrays, margin=0.05):
    val_min = min([float(da.min()) for da in list_of_data_arrays])
    val_max = max([float(da.max()) for da in list_of_data_arrays])
    span = val_max - val_min if val_max != val_min else 1.0
    return (val_min - span * margin, val_max + span * margin)

def format_pearson(r):
    if np.isnan(r):
        return "NaN"
    elif round(r, 2) == 1.0 and r < 1.0:
        return "0.99" 
    elif round(r, 2) == -1.0 and r > -1.0:
        return "-0.99"
    else:
        return f"{r:.2f}"

# =============================================================================
# 3. DATA EXECUTION: ROW 1 (FLUXES)
# =============================================================================
# Baseline Flux
_, vol_u_b_bwd_f, _, vol_e_b_bwd_f, _ = prep_data_flux(file_base_bwd, var_flux_u, var_flux_e)
_, vol_u_b_fwd_f, _, vol_e_b_fwd_f, _ = prep_data_flux(file_base_fwd, var_flux_u, var_flux_e)
dl_lat_b_bwd_f, dl_lon_b_bwd_f = get_1d_diffs_with_pole_filter(vol_u_b_bwd_f, vol_e_b_bwd_f)
dl_lat_b_fwd_f, dl_lon_b_fwd_f = get_1d_diffs_with_pole_filter(vol_u_b_fwd_f, vol_e_b_fwd_f)

# Main Scenario Flux
mm_u_s_bwd_f, vol_u_s_bwd_f, mm_e_s_bwd_f, vol_e_s_bwd_f, a2_bwd_f = prep_data_flux(file_scen_main_bwd, var_flux_u, var_flux_e)
mm_u_s_fwd_f, vol_u_s_fwd_f, mm_e_s_fwd_f, vol_e_s_fwd_f, a2_fwd_f = prep_data_flux(file_scen_main_fwd, var_flux_u, var_flux_e)

diff_map_bwd_f = mm_u_s_bwd_f - mm_e_s_bwd_f
diff_map_fwd_f = mm_u_s_fwd_f - mm_e_s_fwd_f

dl_lat_s_bwd_f, dl_lon_s_bwd_f = get_1d_diffs_with_pole_filter(vol_u_s_bwd_f, vol_e_s_bwd_f)
dl_lat_s_fwd_f, dl_lon_s_fwd_f = get_1d_diffs_with_pole_filter(vol_u_s_fwd_f, vol_e_s_fwd_f)

# Baseline differences calculation
diff_mm_base_bwd_f = (vol_u_b_bwd_f / a2_bwd_f * 1000.0) - (vol_e_b_bwd_f / a2_bwd_f * 1000.0)
diff_mm_base_fwd_f = (vol_u_b_fwd_f / a2_fwd_f * 1000.0) - (vol_e_b_fwd_f / a2_fwd_f * 1000.0)

lim_c_bwd_f = float(np.abs(diff_mm_base_bwd_f).quantile(0.98))
lim_c_fwd_f = float(np.abs(diff_mm_base_fwd_f).quantile(0.98))
limit_diff_flux = max(lim_c_bwd_f, lim_c_fwd_f)
if limit_diff_flux == 0: limit_diff_flux = 1.0 
norm_flux = mcolors.TwoSlopeNorm(vmin=-limit_diff_flux, vcenter=0.0, vmax=limit_diff_flux)

# 1D Flux Limits (including Extra Scenarios)
all_lat_f_bwd = [dl_lat_b_bwd_f, dl_lat_s_bwd_f]
all_lon_f_bwd = [dl_lon_b_bwd_f, dl_lon_s_bwd_f]
all_lat_f_fwd = [dl_lat_b_fwd_f, dl_lat_s_fwd_f]
all_lon_f_fwd = [dl_lon_b_fwd_f, dl_lon_s_fwd_f]

for ex in EXTRA_SCENARIOS:
    _, v_u, _, v_e, _ = prep_data_flux(ex['file_bwd'], var_flux_u, var_flux_e)
    dlat, dlon = get_1d_diffs_with_pole_filter(v_u, v_e)
    ex['diff_lat_bwd_f'] = dlat; ex['diff_lon_bwd_f'] = dlon
    all_lat_f_bwd.append(dlat); all_lon_f_bwd.append(dlon)
    
    _, v_u, _, v_e, _ = prep_data_flux(ex['file_fwd'], var_flux_u, var_flux_e)
    dlat, dlon = get_1d_diffs_with_pole_filter(v_u, v_e)
    ex['diff_lat_fwd_f'] = dlat; ex['diff_lon_fwd_f'] = dlon
    all_lat_f_fwd.append(dlat); all_lon_f_fwd.append(dlon)

xlim_lat_flux = get_multi_envelope_limits(all_lat_f_bwd + all_lat_f_fwd)
ylim_lon_flux = get_multi_envelope_limits(all_lon_f_bwd + all_lon_f_fwd)

# Flux Statistics
st_bwd_f = compute_statistics(mm_u_s_bwd_f, mm_e_s_bwd_f, a2_bwd_f)
st_fwd_f = compute_statistics(mm_u_s_fwd_f, mm_e_s_fwd_f, a2_fwd_f)
v_era_bwd_f = float(vol_e_s_bwd_f.sum().values)
v_era_fwd_f = float(vol_e_s_fwd_f.sum().values)
perc_bwd_f = ((float(vol_u_s_bwd_f.sum().values) - v_era_bwd_f) / v_era_bwd_f) * 100 if v_era_bwd_f > 0 else np.nan
perc_fwd_f = ((float(vol_u_s_fwd_f.sum().values) - v_era_fwd_f) / v_era_fwd_f) * 100 if v_era_fwd_f > 0 else np.nan

# =============================================================================
# 3. DATA EXECUTION: ROW 2 (TCW)
# =============================================================================
# Baseline TCW
_, vol_u_b_bwd_t, _, vol_e_b_bwd_t, _ = prep_data_tcw(file_base_bwd, var_tcw_l, var_tcw_e)
_, vol_u_b_fwd_t, _, vol_e_b_fwd_t, _ = prep_data_tcw(file_base_fwd, var_tcw_l, var_tcw_e)
dl_lat_b_bwd_t, dl_lon_b_bwd_t = get_1d_diffs_with_pole_filter(vol_u_b_bwd_t, vol_e_b_bwd_t)
dl_lat_b_fwd_t, dl_lon_b_fwd_t = get_1d_diffs_with_pole_filter(vol_u_b_fwd_t, vol_e_b_fwd_t)

# Main Scenario TCW
mm_u_s_bwd_t, vol_u_s_bwd_t, mm_e_s_bwd_t, vol_e_s_bwd_t, a2_bwd_t = prep_data_tcw(file_scen_main_bwd, var_tcw_l, var_tcw_e)
mm_u_s_fwd_t, vol_u_s_fwd_t, mm_e_s_fwd_t, vol_e_s_fwd_t, a2_fwd_t = prep_data_tcw(file_scen_main_fwd, var_tcw_l, var_tcw_e)

diff_map_bwd_t = mm_u_s_bwd_t - mm_e_s_bwd_t
diff_map_fwd_t = mm_u_s_fwd_t - mm_e_s_fwd_t

dl_lat_s_bwd_t, dl_lon_s_bwd_t = get_1d_diffs_with_pole_filter(vol_u_s_bwd_t, vol_e_s_bwd_t)
dl_lat_s_fwd_t, dl_lon_s_fwd_t = get_1d_diffs_with_pole_filter(vol_u_s_fwd_t, vol_e_s_fwd_t)

# TCW Color Limits
diff_mm_base_bwd_t = (vol_u_b_bwd_t / a2_bwd_t * 1000.0) - (vol_e_b_bwd_t / a2_bwd_t * 1000.0)
diff_mm_base_fwd_t = (vol_u_b_fwd_t / a2_fwd_t * 1000.0) - (vol_e_b_fwd_t / a2_fwd_t * 1000.0)

lim_c_bwd_t = float(np.abs(diff_mm_base_bwd_t).quantile(0.98))
lim_c_fwd_t = float(np.abs(diff_mm_base_fwd_t).quantile(0.98))
limit_diff_tcw = max(lim_c_bwd_t, lim_c_fwd_t)
if limit_diff_tcw == 0: limit_diff_tcw = 1.0 
norm_tcw = mcolors.TwoSlopeNorm(vmin=-limit_diff_tcw, vcenter=0.0, vmax=limit_diff_tcw)

# 1D TCW Limits
all_lat_t_bwd = [dl_lat_b_bwd_t, dl_lat_s_bwd_t]
all_lon_t_bwd = [dl_lon_b_bwd_t, dl_lon_s_bwd_t]
all_lat_t_fwd = [dl_lat_b_fwd_t, dl_lat_s_fwd_t]
all_lon_t_fwd = [dl_lon_b_fwd_t, dl_lon_s_fwd_t]

for ex in EXTRA_SCENARIOS:
    _, v_u, _, v_e, _ = prep_data_tcw(ex['file_bwd'], var_tcw_l, var_tcw_e)
    dlat, dlon = get_1d_diffs_with_pole_filter(v_u, v_e)
    ex['diff_lat_bwd_t'] = dlat; ex['diff_lon_bwd_t'] = dlon
    all_lat_t_bwd.append(dlat); all_lon_t_bwd.append(dlon)
    
    _, v_u, _, v_e, _ = prep_data_tcw(ex['file_fwd'], var_tcw_l, var_tcw_e)
    dlat, dlon = get_1d_diffs_with_pole_filter(v_u, v_e)
    ex['diff_lat_fwd_t'] = dlat; ex['diff_lon_fwd_t'] = dlon
    all_lat_t_fwd.append(dlat); all_lon_t_fwd.append(dlon)

xlim_lat_tcw = get_multi_envelope_limits(all_lat_t_bwd + all_lat_t_fwd)
ylim_lon_tcw = get_multi_envelope_limits(all_lon_t_bwd + all_lon_t_fwd)

# TCW Statistics
st_bwd_t = compute_statistics(mm_u_s_bwd_t, mm_e_s_bwd_t, a2_bwd_t)
st_fwd_t = compute_statistics(mm_u_s_fwd_t, mm_e_s_fwd_t, a2_fwd_t)
v_era_bwd_t = float(vol_e_s_bwd_t.sum().values)
v_era_fwd_t = float(vol_e_s_fwd_t.sum().values)
perc_bwd_t = ((float(vol_u_s_bwd_t.sum().values) - v_era_bwd_t) / v_era_bwd_t) * 100 if v_era_bwd_t > 0 else np.nan
perc_fwd_t = ((float(vol_u_s_fwd_t.sum().values) - v_era_fwd_t) / v_era_fwd_t) * 100 if v_era_fwd_t > 0 else np.nan

# =============================================================================
# 4. COLORMAPS & SCALERS 
# =============================================================================
cmap_diff = cmocean.cm.diff_r.copy() 
COLOR_POS = cmap_diff(0.85) 
COLOR_NEG = cmap_diff(0.15) 

scale_zon_f = 1e10; lab_z_f = "10^1"
scale_mer_f = 1e9;  #lab_m_f = "10^9"

scale_zon_t = 1e9;  #lab_z_t = "10^9"
scale_mer_t = 1e9;  #lab_m_t = "10^8"

# =============================================================================
# 5. 4-PANEL HORIZONTAL LAYOUT (A4 LANDSCAPE)
# =============================================================================
fig_w, fig_h = 20.0, 12.0 
fig = plt.figure(figsize=(fig_w, fig_h)) 
proj = ccrs.PlateCarree()

w_map = 0.32            
h_map = (w_map * fig_w) / (2.0 * fig_h) 
w_zon = 0.08            
h_mer = 0.1             
h_cbar = 0.015          

# Horizontal Coordinates
x_left_c1 = 0.05
x_right_c1 = x_left_c1 + w_map + 0.01 
gap_cols = 0.08
x_left_c2 = x_right_c1 + w_zon + gap_cols
x_right_c2 = x_left_c2 + w_map + 0.01 

# Vertical Coordinates
gap_map_mer = 0.01    
gap_mer_cbar = 0.05   
gap_blocks = 0.06    

# Top (Row 1 - Fluxes)
y1_map_t = 0.95
y1_map_b = y1_map_t - h_map
y1_mer_t = y1_map_b - gap_map_mer
y1_mer_b = y1_mer_t - h_mer
y1_cbar_t = y1_mer_b - gap_mer_cbar
y1_cbar_b = y1_cbar_t - h_cbar

# Bottom (Row 2 - TCW)
y2_map_t = y1_cbar_b - gap_blocks
y2_map_b = y2_map_t - h_map
y2_mer_t = y2_map_b - gap_map_mer
y2_mer_b = y2_mer_t - h_mer
y2_cbar_t = y2_mer_b - gap_mer_cbar
y2_cbar_b = y2_cbar_t - h_cbar

y1_box_top = y1_map_b - 0.055
h_box1 = y1_box_top - y1_mer_b
y2_box_top = y2_map_b - 0.055
h_box2 = y2_box_top - y2_mer_b

# ===========================================================================
# BLOCK 1A: ROW 1, COL 1 (FLUX DIFFERENCE - BWD)
# ===========================================================================
ax_m1 = fig.add_axes([x_left_c1, y1_map_b, w_map, h_map], projection=proj)
ax_m1.set_rasterization_zorder(0)
im1 = ax_m1.pcolormesh(diff_map_bwd_f.lon, diff_map_bwd_f.lat, diff_map_bwd_f, transform=proj, cmap=cmap_diff, norm=norm_flux, zorder=-1)
ax_m1.coastlines(linewidth=0.5, zorder=1)
ax_m1.set_extent([-180, 180, -90, 90], crs=proj)
ax_m1.text(-0.07, 1.0, 'a', transform=ax_m1.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m1.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z1 = fig.add_axes([x_right_c1, y1_map_b, w_zon, h_map])
ax_z1.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_z1.plot(dl_lat_b_bwd_f / scale_zon_f, dl_lat_b_bwd_f.lat, color='#999999', linestyle='--', linewidth=1.5, zorder=2, label=LABEL_BASE)
for ex in EXTRA_SCENARIOS:
    ax_z1.plot(ex['diff_lat_bwd_f'] / scale_zon_f, ex['diff_lat_bwd_f'].lat, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3, label=ex['label'])
dl_lat_s_bwd_f_scl = dl_lat_s_bwd_f / scale_zon_f
ax_z1.fill_betweenx(dl_lat_s_bwd_f_scl.lat, 0, dl_lat_s_bwd_f_scl, where=(dl_lat_s_bwd_f_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_z1.fill_betweenx(dl_lat_s_bwd_f_scl.lat, 0, dl_lat_s_bwd_f_scl, where=(dl_lat_s_bwd_f_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_z1.plot(dl_lat_s_bwd_f_scl, dl_lat_s_bwd_f_scl.lat, color='black', linewidth=1.8, zorder=5, label=LABEL_SCEN_MAIN)
ax_z1.set_xlim([x / scale_zon_f for x in xlim_lat_flux]) 
ax_z1.set_ylim(-90, 90)
ax_z1.set_yticks(np.arange(-90, 91, 30))
ax_z1.yaxis.tick_right()
ax_z1.yaxis.set_label_position("right")
ax_z1.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z1.set_ylabel("Latitude")
ax_z1.set_xlabel(f"$\\Delta$ Volume (${lab_z_f}$ km$^3$)")
ax_z1.xaxis.set_major_formatter(formatter_1d)
ax_z1.grid(True, linestyle=':', alpha=0.6)
ax_z1.spines['top'].set_visible(False)
ax_z1.spines['left'].set_visible(False)

ax_me1 = fig.add_axes([x_left_c1, y1_mer_b, w_map, h_mer])
ax_me1.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_me1.plot(dl_lon_b_bwd_f.lon, dl_lon_b_bwd_f / scale_mer_f, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_me1.plot(ex['diff_lon_bwd_f'].lon, ex['diff_lon_bwd_f'] / scale_mer_f, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lon_s_bwd_f_scl = dl_lon_s_bwd_f / scale_mer_f
ax_me1.fill_between(dl_lon_s_bwd_f_scl.lon, 0, dl_lon_s_bwd_f_scl, where=(dl_lon_s_bwd_f_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_me1.fill_between(dl_lon_s_bwd_f_scl.lon, 0, dl_lon_s_bwd_f_scl, where=(dl_lon_s_bwd_f_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_me1.plot(dl_lon_s_bwd_f_scl.lon, dl_lon_s_bwd_f_scl, color='black', linewidth=1.8, zorder=5)
ax_me1.set_ylim([y / scale_mer_f for y in ylim_lon_flux])
ax_me1.set_xlim(-180, 180)
ax_me1.set_xticks(np.arange(-180, 181, 60))
ax_me1.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me1.set_ylabel(f"$\\Delta$ Volume (km$^3$)")
ax_me1.set_xlabel("Longitude")
ax_me1.yaxis.set_major_formatter(formatter_1d)
ax_me1.grid(True, linestyle=':', alpha=0.6)
ax_me1.spines['top'].set_visible(False)
ax_me1.spines['right'].set_visible(False)

# Legend BWD Row 1
handles, labels = ax_z1.get_legend_handles_labels()
label_idx = {label: i for i, label in enumerate(labels)}
ordered_labels = [LABEL_SCEN_MAIN, LABEL_BASE]
for ex in EXTRA_SCENARIOS: ordered_labels.append(ex['label'])
ordered_handles = [handles[label_idx[lbl]] for lbl in ordered_labels]
ax_me1.legend(ordered_handles, ordered_labels, loc='upper left', ncol=4, fontsize=11, frameon=False)

ax_st1 = fig.add_axes([x_right_c1+0.01, y1_mer_b-0.02, w_zon-0.005, h_box1]) 
ax_st1.axis('off')
txt_1 = (f"Pearson $r$: {st_bwd_f['r']:.2f}\nRMSE: {st_bwd_f['RMSE']:.2f} mm\nMAE: {st_bwd_f['MAE']:.2f} mm\nKGE: {st_bwd_f['KGE']:.2f}")
ax_st1.text(0.05, 0.95, txt_1, transform=ax_st1.transAxes, ha='left', va='top', fontsize=12)

cb_ax1 = fig.add_axes([x_left_c1, y1_cbar_b, w_map, h_cbar])
cb1 = fig.colorbar(im1, cax=cb_ax1, orientation='horizontal', extend='both')
cb1.solids.set_rasterized(True); cb1.solids.set_edgecolor('face'); cb1.outline.set_edgecolor('black')
cb1.set_label(f'ET Difference (UTrack - {LABEL_ERA5}) [mm]', weight='bold')

# ===========================================================================
# BLOCK 1B: ROW 1, COL 2 (FLUX DIFFERENCE - FWD)
# ===========================================================================
ax_m2 = fig.add_axes([x_left_c2, y1_map_b, w_map, h_map], projection=proj)
ax_m2.set_rasterization_zorder(0)
im2 = ax_m2.pcolormesh(diff_map_fwd_f.lon, diff_map_fwd_f.lat, diff_map_fwd_f, transform=proj, cmap=cmap_diff, norm=norm_flux, zorder=-1)
ax_m2.coastlines(linewidth=0.5, zorder=1)
ax_m2.set_extent([-180, 180, -90, 90], crs=proj)
ax_m2.text(-0.07, 1.0, 'b', transform=ax_m2.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m2.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z2 = fig.add_axes([x_right_c2, y1_map_b, w_zon, h_map])
ax_z2.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_z2.plot(dl_lat_b_fwd_f / scale_zon_f, dl_lat_b_fwd_f.lat, color='#999999', linestyle='--', linewidth=1.5, zorder=2, label=LABEL_BASE)
for ex in EXTRA_SCENARIOS:
    ax_z2.plot(ex['diff_lat_fwd_f'] / scale_zon_f, ex['diff_lat_fwd_f'].lat, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3, label=ex['label'])
dl_lat_s_fwd_f_scl = dl_lat_s_fwd_f / scale_zon_f
ax_z2.fill_betweenx(dl_lat_s_fwd_f_scl.lat, 0, dl_lat_s_fwd_f_scl, where=(dl_lat_s_fwd_f_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_z2.fill_betweenx(dl_lat_s_fwd_f_scl.lat, 0, dl_lat_s_fwd_f_scl, where=(dl_lat_s_fwd_f_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_z2.plot(dl_lat_s_fwd_f_scl, dl_lat_s_fwd_f_scl.lat, color='black', linewidth=1.8, zorder=5, label=LABEL_SCEN_MAIN)
ax_z2.set_xlim([x / scale_zon_f for x in xlim_lat_flux]) 
ax_z2.set_ylim(-90, 90)
ax_z2.set_yticks(np.arange(-90, 91, 30))
ax_z2.yaxis.tick_right()
ax_z2.yaxis.set_label_position("right")
ax_z2.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z2.set_ylabel("Latitude")
ax_z2.set_xlabel(f"$\\Delta$ Volume (${lab_z_f}$ km$^3$)")
ax_z2.xaxis.set_major_formatter(formatter_1d)
ax_z2.grid(True, linestyle=':', alpha=0.6)
ax_z2.spines['top'].set_visible(False)
ax_z2.spines['left'].set_visible(False)

ax_me2 = fig.add_axes([x_left_c2, y1_mer_b, w_map, h_mer])
ax_me2.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_me2.plot(dl_lon_b_fwd_f.lon, dl_lon_b_fwd_f / scale_mer_f, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_me2.plot(ex['diff_lon_fwd_f'].lon, ex['diff_lon_fwd_f'] / scale_mer_f, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lon_s_fwd_f_scl = dl_lon_s_fwd_f / scale_mer_f
ax_me2.fill_between(dl_lon_s_fwd_f_scl.lon, 0, dl_lon_s_fwd_f_scl, where=(dl_lon_s_fwd_f_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_me2.fill_between(dl_lon_s_fwd_f_scl.lon, 0, dl_lon_s_fwd_f_scl, where=(dl_lon_s_fwd_f_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_me2.plot(dl_lon_s_fwd_f_scl.lon, dl_lon_s_fwd_f_scl, color='black', linewidth=1.8, zorder=5)
ax_me2.set_ylim([y / scale_mer_f for y in ylim_lon_flux])
ax_me2.set_xlim(-180, 180)
ax_me2.set_xticks(np.arange(-180, 181, 60))
ax_me2.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me2.set_ylabel(f"$\\Delta$ Volume (km$^3$)")
ax_me2.set_xlabel("Longitude")
ax_me2.yaxis.set_major_formatter(formatter_1d)
ax_me2.grid(True, linestyle=':', alpha=0.6)
ax_me2.spines['top'].set_visible(False)
ax_me2.spines['right'].set_visible(False)

ax_me2.legend(ordered_handles, ordered_labels, loc='lower left', ncol=4, fontsize=11, frameon=False)

ax_st2 = fig.add_axes([x_right_c2+0.01, y1_mer_b-0.02, w_zon-0.01, h_box1]) 
ax_st2.axis('off')
txt_2 = (f"Pearson $r$: {format_pearson(st_fwd_f['r'])}\nRMSE: {st_fwd_f['RMSE']:.2f} mm\nMAE: {st_fwd_f['MAE']:.2f} mm\nKGE: {st_fwd_f['KGE']:.2f}")
ax_st2.text(0.05, 0.95, txt_2, transform=ax_st2.transAxes, ha='left', va='top', fontsize=12)

cb_ax2 = fig.add_axes([x_left_c2, y1_cbar_b, w_map, h_cbar])
cb2 = fig.colorbar(im2, cax=cb_ax2, orientation='horizontal', extend='both')
cb2.solids.set_rasterized(True); cb2.solids.set_edgecolor('face'); cb2.outline.set_edgecolor('black')
cb2.set_label(f'Precipitation Difference (UTrack - {LABEL_ERA5}) [mm]', weight='bold')

# ===========================================================================
# BLOCK 2A: ROW 2, COL 1 (TCW DIFFERENCE - BWD)
# ===========================================================================
ax_m3 = fig.add_axes([x_left_c1, y2_map_b, w_map, h_map], projection=proj)
ax_m3.set_rasterization_zorder(0)
im3 = ax_m3.pcolormesh(diff_map_bwd_t.lon, diff_map_bwd_t.lat, diff_map_bwd_t, transform=proj, cmap=cmap_diff, norm=norm_tcw, zorder=-1)
ax_m3.coastlines(linewidth=0.5, zorder=1)
ax_m3.set_extent([-180, 180, -90, 90], crs=proj)
ax_m3.text(-0.07, 1.0, 'c', transform=ax_m3.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m3.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z3 = fig.add_axes([x_right_c1, y2_map_b, w_zon, h_map])
ax_z3.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_z3.plot(dl_lat_b_bwd_t / scale_zon_t, dl_lat_b_bwd_t.lat, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_z3.plot(ex['diff_lat_bwd_t'] / scale_zon_t, ex['diff_lat_bwd_t'].lat, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lat_s_bwd_t_scl = dl_lat_s_bwd_t / scale_zon_t
ax_z3.fill_betweenx(dl_lat_s_bwd_t_scl.lat, 0, dl_lat_s_bwd_t_scl, where=(dl_lat_s_bwd_t_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_z3.fill_betweenx(dl_lat_s_bwd_t_scl.lat, 0, dl_lat_s_bwd_t_scl, where=(dl_lat_s_bwd_t_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_z3.plot(dl_lat_s_bwd_t_scl, dl_lat_s_bwd_t_scl.lat, color='black', linewidth=1.8, zorder=5)
ax_z3.set_xlim([x / scale_zon_t for x in xlim_lat_tcw]) 
ax_z3.set_ylim(-90, 90)
ax_z3.set_yticks(np.arange(-90, 91, 30))
ax_z3.yaxis.tick_right()
ax_z3.yaxis.set_label_position("right")
ax_z3.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z3.set_ylabel("Latitude")
ax_z3.set_xlabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_z3.xaxis.set_major_formatter(formatter_1d)
ax_z3.grid(True, linestyle=':', alpha=0.6)
ax_z3.spines['top'].set_visible(False)
ax_z3.spines['left'].set_visible(False)

ax_me3 = fig.add_axes([x_left_c1, y2_mer_b, w_map, h_mer])
ax_me3.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_me3.plot(dl_lon_b_bwd_t.lon, dl_lon_b_bwd_t / scale_mer_t, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_me3.plot(ex['diff_lon_bwd_t'].lon, ex['diff_lon_bwd_t'] / scale_mer_t, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lon_s_bwd_t_scl = dl_lon_s_bwd_t / scale_mer_t
ax_me3.fill_between(dl_lon_s_bwd_t_scl.lon, 0, dl_lon_s_bwd_t_scl, where=(dl_lon_s_bwd_t_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_me3.fill_between(dl_lon_s_bwd_t_scl.lon, 0, dl_lon_s_bwd_t_scl, where=(dl_lon_s_bwd_t_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_me3.plot(dl_lon_s_bwd_t_scl.lon, dl_lon_s_bwd_t_scl, color='black', linewidth=1.8, zorder=5)
ax_me3.set_ylim([y / scale_mer_t for y in ylim_lon_tcw])
ax_me3.set_xlim(-180, 180)
ax_me3.set_xticks(np.arange(-180, 181, 60))
ax_me3.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me3.set_ylabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_me3.set_xlabel("Longitude")
ax_me3.yaxis.set_major_formatter(formatter_1d)
ax_me3.grid(True, linestyle=':', alpha=0.6)
ax_me3.spines['top'].set_visible(False)
ax_me3.spines['right'].set_visible(False)

ax_me3.legend(ordered_handles, ordered_labels, loc='upper left', ncol=4, fontsize=11, frameon=False)

ax_st3 = fig.add_axes([x_right_c1+0.01, y2_mer_b-0.02, w_zon-0.005, h_box2]) 
ax_st3.axis('off')
txt_3 = (f"Pearson $r$: {st_bwd_t['r']:.2f}\nRMSE: {st_bwd_t['RMSE']:.2f} mm/h\nMAE: {st_bwd_t['MAE']:.2f} mm/h\nKGE: {st_bwd_t['KGE']:.2f}")
ax_st3.text(0.05, 0.95, txt_3, transform=ax_st3.transAxes, ha='left', va='top', fontsize=12)

cb_ax3 = fig.add_axes([x_left_c1, y2_cbar_b, w_map, h_cbar])
cb3 = fig.colorbar(im3, cax=cb_ax3, orientation='horizontal', extend='both')
cb3.solids.set_rasterized(True); cb3.solids.set_edgecolor('face'); cb3.outline.set_edgecolor('black')
cb3.set_label(f'TCW Difference (UTrack - {LABEL_ERA5}) [mm/h]', weight='bold')

# ===========================================================================
# BLOCK 2B: ROW 2, COL 2 (TCW DIFFERENCE - FWD)
# ===========================================================================
ax_m4 = fig.add_axes([x_left_c2, y2_map_b, w_map, h_map], projection=proj)
ax_m4.set_rasterization_zorder(0)
im4 = ax_m4.pcolormesh(diff_map_fwd_t.lon, diff_map_fwd_t.lat, diff_map_fwd_t, transform=proj, cmap=cmap_diff, norm=norm_tcw, zorder=-1)
ax_m4.coastlines(linewidth=0.5, zorder=1)
ax_m4.set_extent([-180, 180, -90, 90], crs=proj)
ax_m4.text(-0.07, 1.0, 'd', transform=ax_m4.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m4.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z4 = fig.add_axes([x_right_c2, y2_map_b, w_zon, h_map])
ax_z4.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_z4.plot(dl_lat_b_fwd_t / scale_zon_t, dl_lat_b_fwd_t.lat, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_z4.plot(ex['diff_lat_fwd_t'] / scale_zon_t, ex['diff_lat_fwd_t'].lat, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lat_s_fwd_t_scl = dl_lat_s_fwd_t / scale_zon_t
ax_z4.fill_betweenx(dl_lat_s_fwd_t_scl.lat, 0, dl_lat_s_fwd_t_scl, where=(dl_lat_s_fwd_t_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_z4.fill_betweenx(dl_lat_s_fwd_t_scl.lat, 0, dl_lat_s_fwd_t_scl, where=(dl_lat_s_fwd_t_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_z4.plot(dl_lat_s_fwd_t_scl, dl_lat_s_fwd_t_scl.lat, color='black', linewidth=1.8, zorder=5)
ax_z4.set_xlim([x / scale_zon_t for x in xlim_lat_tcw]) 
ax_z4.set_ylim(-90, 90)
ax_z4.set_yticks(np.arange(-90, 91, 30))
ax_z4.yaxis.tick_right()
ax_z4.yaxis.set_label_position("right")
ax_z4.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z4.set_ylabel("Latitude") 
ax_z4.set_xlabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_z4.xaxis.set_major_formatter(formatter_1d)
ax_z4.grid(True, linestyle=':', alpha=0.6)
ax_z4.spines['top'].set_visible(False)
ax_z4.spines['left'].set_visible(False)

ax_me4 = fig.add_axes([x_left_c2, y2_mer_b, w_map, h_mer])
ax_me4.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
ax_me4.plot(dl_lon_b_fwd_t.lon, dl_lon_b_fwd_t / scale_mer_t, color='#999999', linestyle='--', linewidth=1.5, zorder=2)
for ex in EXTRA_SCENARIOS:
    ax_me4.plot(ex['diff_lon_fwd_t'].lon, ex['diff_lon_fwd_t'] / scale_mer_t, color=ex['color'], linestyle=ex['linestyle'], linewidth=1.2, zorder=3)
dl_lon_s_fwd_t_scl = dl_lon_s_fwd_t / scale_mer_t
ax_me4.fill_between(dl_lon_s_fwd_t_scl.lon, 0, dl_lon_s_fwd_t_scl, where=(dl_lon_s_fwd_t_scl>0), color=COLOR_POS, alpha=0.5, zorder=4)
ax_me4.fill_between(dl_lon_s_fwd_t_scl.lon, 0, dl_lon_s_fwd_t_scl, where=(dl_lon_s_fwd_t_scl<0), color=COLOR_NEG, alpha=0.5, zorder=4)
ax_me4.plot(dl_lon_s_fwd_t_scl.lon, dl_lon_s_fwd_t_scl, color='black', linewidth=1.8, zorder=5)
ax_me4.set_ylim([y / scale_mer_t for y in ylim_lon_tcw])
ax_me4.set_xlim(-180, 180)
ax_me4.set_xticks(np.arange(-180, 181, 60))
ax_me4.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me4.set_ylabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_me4.set_xlabel("Longitude")
ax_me4.yaxis.set_major_formatter(formatter_1d)
ax_me4.grid(True, linestyle=':', alpha=0.6)
ax_me4.spines['top'].set_visible(False)
ax_me4.spines['right'].set_visible(False)

ax_me4.legend(ordered_handles, ordered_labels, loc='lower left', ncol=4, fontsize=11, frameon=False)

ax_st4 = fig.add_axes([x_right_c2+0.01, y2_mer_b-0.02, w_zon-0.01, h_box2]) 
ax_st4.axis('off')
txt_4 = (f"Pearson $r$: {format_pearson(st_fwd_t['r'])}\nRMSE: {st_fwd_t['RMSE']:.2f} mm/h\nMAE: {st_fwd_t['MAE']:.2f} mm/h\nKGE: {st_fwd_t['KGE']:.2f}")
ax_st4.text(0.05, 0.95, txt_4, transform=ax_st4.transAxes, ha='left', va='top', fontsize=12)

cb_ax4 = fig.add_axes([x_left_c2, y2_cbar_b, w_map, h_cbar])
cb4 = fig.colorbar(im4, cax=cb_ax4, orientation='horizontal', extend='both')
cb4.solids.set_rasterized(True); cb4.solids.set_edgecolor('face'); cb4.outline.set_edgecolor('black')
cb4.set_label(f'TCW Difference (UTrack - {LABEL_ERA5}) [mm/h]', weight='bold')

plt.savefig('fig3.pdf', format='pdf', bbox_inches='tight', dpi=600)
plt.show()